# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinivas25046/FlyRank-MLstarter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Raw grain:** one row in `fact_content_daily_performance` = one **(report_date, client, content)**
observation — one page's activity, for one client, on one day.

**Analysis grain (what my lane actually ranks):** one row = one **(client, content)** pair,
described by a trailing feature window and, for anything forward-looking, a separate later
outcome window. For this contract I develop and verify everything on a **mid-panel month**
(`month = 2026-02` for features, `month = 2026-03` as the following month) — never on the
sealed final month (`2026-06`), which is the natural answer key for any past→future label and
is reserved as a held-out test month, not for building label logic.

I verify the raw grain, the row count/date span, and availability below before trusting
anything built on top of it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


%pip -q install duckdb

import duckdb
from getpass import getpass

# --- Auth: register the HF read token as a DuckDB secret. Never paste a token into a cell ---
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("Hugging Face READ token (from a Colab Secret named HF_TOKEN ideally): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# --- Table paths. Check the dataset's "Files" tab on huggingface.co/datasets/FlyRank/internship-warehouse
#     to confirm single-file vs folder shape before trusting these -- see querying-big-datasets/SKILL.md ---
BASE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"{BASE}/dim_clients.parquet"
DIM_CONTENT = f"{BASE}/dim_content.parquet"
DAILY_FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"  # partitioned by month=YYYY-MM

FEATURE_MONTH = "2026-02"  # mid-panel month used to build features
LABEL_MONTH = "2026-03"    # the following month, used only to check a forward outcome

# See the REAL column names before writing a single query against them.
schema_df = con.sql(f"""
    SELECT * FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
    LIMIT 0
""").df()
print("fact_content_daily_performance columns:")
print(list(schema_df.columns))

fact_content_daily_performance columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (known at the end of the feature month, before the label month exists):
`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `sessions_ai` — all summed
or averaged over `FEATURE_MONTH` only. Content metadata from `dim_content` (`word_count`,
`content_type`, age) is a feature too, once joined on `content_hash_id`.

**Label / proxy:** `declined_next_month` — built by comparing `LABEL_MONTH` impressions back
against `FEATURE_MONTH` impressions (a real forward-looking outcome, not a same-window bucket
like the starter CSV's `trend_direction`). Never a feature.

**Context** (grouping/joining/splitting only, never model input): `client_hash_id`,
`content_hash_id`, `report_date`, `url_hash_id`, `keyword_hash_id`.

**Excluded, each with a why:**
- Any FlyRank product-decision field (`health_score`, `priority_score`, `action_type`,
  refresh flags) — the lane guide states these are rule-outputs FlyRank's app computes and are
  deliberately **not shipped** in this release; there's nothing to strip, but if I ever rebuild
  one myself it's a baseline to beat, never a feature.
- Rows before a client's `gsc_data_start` / `ga4_data_start` — zero-filled, not real zero
  activity; excluded via the availability flags (query 3 below), not by treating them as data.
- The sealed final month (`2026-06` / `_sample` table) — excluded from all feature and label
  *development*; it's the natural outcome window for any past→future label and stays a sealed
  test month.
- Raw-origin hash columns (`url_hash_id`, `keyword_hash_id`) are context for grouping only —
  never treated as if they revealed the underlying query/URL, since the originals were
  scrambled before release.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Cross-check my declared feature/context columns actually exist in the real schema above.
declared = {
    "gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_sessions", "sessions_ai",
    "client_hash_id", "content_hash_id", "report_date",
}
missing = declared - set(schema_df.columns)
print("Declared columns missing from the real schema (should be empty):", missing or "none")

Declared columns missing from the real schema (should be empty): none


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks against `FEATURE_MONTH` (2026-02): the grain holds, the row count/date span
matches what the docs promise, and how many rows survive an honest `IS TRUE` availability
filter (not `= FALSE`, which quietly drops the NULLs the flyrank-data skill warns about).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("== Query 1: grain probe -- one row really is one (date, client, content)? ==")
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Rows violating the grain: {len(grain_check)} (should be 0)")

print("\n== Query 2: row count + date span for month =", FEATURE_MONTH, "==")
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
""").df()
print(counts.to_string(index=False))

print("\n== Query 3: availability -- filter with IS TRUE, show survival rate ==")
# NOTE: gsc_data_available's exact name is a guess from the flyrank-data skill's mention of
# "the GSC flag" alongside ga4_data_available -- confirm against the printed schema above.
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
""").df()
avail["ga4_available_pct"] = avail["ga4_available_rows"] / avail["total_rows"] * 100
avail["gsc_available_pct"] = avail["gsc_available_rows"] / avail["total_rows"] * 100
print(avail.to_string(index=False))

== Query 1: grain probe -- one row really is one (date, client, content)? ==


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the grain: 0 (should be 0)

== Query 2: row count + date span for month = 2026-02 ==
 n_rows   min_date   max_date  n_clients  n_content
7355108 2026-02-01 2026-02-28         54     321546

== Query 3: availability -- filter with IS TRUE, show survival rate ==


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  ga4_available_rows  gsc_available_rows  ga4_available_pct  gsc_available_pct
    7355108            145321.0           2621783.0           1.975783          35.645744


**Five features, max** — each knowable at the end of `FEATURE_MONTH`, before `LABEL_MONTH`
exists:

| Feature | Knowable at the decision moment because... |
|---|---|
| `feb_impressions` | it's a sum over the feature month only, measured before March starts |
| `feb_clicks` | same — trailing, not forward-looking |
| `feb_avg_position` | GSC position for February, already settled by month's end |
| `feb_sessions` | GA4 sessions for February only |
| `feb_ai_sessions` | AI-referred sessions for February only (thin signal, but still prior-window) |

In [5]:
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS feb_impressions,
           SUM(gsc_clicks)      AS feb_clicks,
           AVG(gsc_avg_position) AS feb_avg_position,
           SUM(ga4_sessions)     AS feb_sessions,
           SUM(sessions_ai)      AS feb_ai_sessions
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"(client, content) pairs with February activity: {len(feat):,}")
feat.head(5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(client, content) pairs with February activity: 321,546


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,feb_sessions,feb_ai_sessions
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,29.609070,5.0,0.0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,17.806923,13.0,0.0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,7.852945,1.0,0.0
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,7.123694,1.0,0.0
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,7.965842,0.0,0.0


**The trap:** add one label-derived column on purpose. The label I'm about to define
(`declined_next_month`) is a direct comparison of March impressions against February
impressions — so if March's own impressions leak into the "features," a trivial rule can
reconstruct the label almost perfectly. Watch the score jump, then delete the column and
keep the honest number.

In [6]:
label = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS mar_impressions
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{LABEL_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

panel = feat.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
panel = panel[panel["feb_impressions"] > 0].copy()  # avoid divide-by-zero noise
panel["declined_next_month"] = (
    panel["mar_impressions"] < panel["feb_impressions"] * 0.8
).astype(int)
print(f"(client, content) pairs with both months activity: {len(panel):,}")
print(f"Share that declined next month: {panel['declined_next_month'].mean():.1%}")

# --- THE LEAK: put March's own impressions in as if it were a "feature" ---
panel["LEAKY_mar_impressions"] = panel["mar_impressions"]
panel["quick_score_LEAKY"] = (
    panel["LEAKY_mar_impressions"] < panel["feb_impressions"] * 0.8
).astype(int)
leaky_accuracy = (panel["quick_score_LEAKY"] == panel["declined_next_month"]).mean()
print(f"\n'Accuracy' using the leaky column directly: {leaky_accuracy:.1%}  <- suspiciously perfect")
print("Not a coincidence -- the label IS this column compared to February. Including it")
print("doesn't predict the future, it just restates the definition of the label.")

# --- Delete the leak. Check honestly, using ONLY February-internal signal, no March at all ---
panel = panel.drop(columns=["LEAKY_mar_impressions", "quick_score_LEAKY"])

feb_halves = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date < DATE '2026-02-15' THEN gsc_impressions ELSE 0 END) AS feb_h1,
           SUM(CASE WHEN report_date >= DATE '2026-02-15' THEN gsc_impressions ELSE 0 END) AS feb_h2
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

panel = panel.merge(feb_halves, on=["client_hash_id", "content_hash_id"], how="left")
panel["honest_score"] = (panel["feb_h2"] < panel["feb_h1"] * 0.8).astype(int)
honest_accuracy = (panel["honest_score"] == panel["declined_next_month"]).mean()
print(f"\nHonest accuracy, using only Feb-internal trend (no March at all): {honest_accuracy:.1%}")
print("-> That gap between the two numbers IS the leakage lesson, performed on real")
print("   warehouse data instead of just narrated.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(client, content) pairs with both months activity: 145,279
Share that declined next month: 26.1%

'Accuracy' using the leaky column directly: 100.0%  <- suspiciously perfect
Not a coincidence -- the label IS this column compared to February. Including it
doesn't predict the future, it just restates the definition of the label.

Honest accuracy, using only Feb-internal trend (no March at all): 70.3%
-> That gap between the two numbers IS the leakage lesson, performed on real
   warehouse data instead of just narrated.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced panel.** Per-client history depth differs wildly (`dim_clients.gsc_data_start`
  / `ga4_data_start` — checked below). A global calendar window silently favors clients with
  longer history; per-client windows are the honest default.
- **GSC-only early history.** Rows before a client's `ga4_data_start` have GA4 columns
  zero-filled with `ga4_data_available = FALSE` (and sometimes NULL, never a plain `FALSE`) —
  those zeros mean "not measured," not "no engagement." Filtering `= FALSE` instead of
  `IS TRUE` on the complement quietly miscounts, since NULL is neither.
- **Mid-window registration.** Content items registered partway through a window only accrue
  daily-fact history from their registration day forward — the data dictionary notes ~98.5%
  reconciliation against the query table for exactly this reason. Absent history there means
  "not yet tracked," not "zero activity."
- **Query-table window overlap.** `fact_content_query_90d` covers a fixed 90-day window that
  overlaps the snapshot's final months. If a label lives in the last 30 days, only its
  `*_prev30`-style columns are safe features — this contract avoids the query table entirely
  for that reason, sticking to the daily fact table's own trailing windows instead.
- **The sealed test month.** `2026-06` (and `_sample`) is the natural outcome window for any
  past→future label defined here — it's excluded from every step above, reserved as a final
  check, not a development month.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

hist = con.sql(f"""
    SELECT gsc_data_start, ga4_data_start
    FROM read_parquet('{DIM_CLIENTS}')
    ORDER BY gsc_data_start
""").df()

print(f"Distinct clients: {len(hist):,}")
print("\nEarliest / latest gsc_data_start:", hist["gsc_data_start"].min(), "/", hist["gsc_data_start"].max())
print("Earliest / latest ga4_data_start:", hist["ga4_data_start"].min(), "/", hist["ga4_data_start"].max())
print("\n-> Confirms the unbalanced panel directly: history depth is not the same across clients,")
print("   so a single global time window would treat 'not tracked yet' as 'zero activity' for some.")

Distinct clients: 104

Earliest / latest gsc_data_start: 2025-01-27 00:00:00 / 2026-06-02 00:00:00
Earliest / latest ga4_data_start: 2025-10-29 00:00:00 / 2026-06-01 00:00:00

-> Confirms the unbalanced panel directly: history depth is not the same across clients,
   so a single global time window would treat 'not tracked yet' as 'zero activity' for some.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.